In [13]:
!pip install -q \
youtube-transcript-api \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain-google-genai \
faiss-cpu \
sentence-transformers

In [14]:
from getpass import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API Key: ")

Enter Gemini API Key: ··········


In [15]:
from youtube_transcript_api import YouTubeTranscriptApi

from urllib.parse import urlparse, parse_qs

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI

In [16]:
def extract_video_id(url):
    parsed_url = urlparse(url)

    if parsed_url.hostname == "youtu.be":
        return parsed_url.path[1:]

    if parsed_url.hostname in [
        "youtube.com",
        "www.youtube.com"
    ]:
        return parse_qs(parsed_url.query)["v"][0]

    return None


youtube_url = input("Enter YouTube URL: ")

video_id = extract_video_id(youtube_url)

print("Video ID:", video_id)

Enter YouTube URL: https://www.youtube.com/watch?v=J5_-l7WIO_w&t=538s
Video ID: J5_-l7WIO_w


In [17]:
ytt_api = YouTubeTranscriptApi()

try:
    transcript = ytt_api.fetch(
        video_id,
        languages=["en"]
    )
except:
    transcript = ytt_api.fetch(
        video_id,
        languages=["hi"]
    )

transcript_text = " ".join(
    [item.text for item in transcript]
)

print(transcript_text[:500])

हाय गाइज़, माय नेम इज नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू करेंगे। अह पिछले वीडियो में हमने रैग पढ़ना शुरू किया था और हमने फोकस किया था रैग के अराउंड जो भी थ्योरी है उसको डिस्कस करने के ऊपर। मैंने आपको बताया था कि रैग क्या होता है? उसकी जरूरत क्यों होती है? मैंने वहां पर रैक को कंपेयर करके भी दिखाया था फाइन ट्यूनिंग जैसी टेक्निक के साथ। और आज का जो वीडियो है वह पिछले वीडियो का ही कंटिन्यूएशन है। जहां पर हम प्रैक्टिकली एक रैग बेस्ड सिस्


In [18]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents(
    [transcript_text]
)

print("Total Chunks:", len(chunks))

Total Chunks: 51


In [19]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

print("Vector Store Created")

Vector Store Created


In [21]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever Created")

Retriever Created


In [22]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("LLM Ready")

LLM Ready


In [23]:
def ask_video(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
    Answer only from the provided context.

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    return response.content

In [ ]:
while True:

    question = input("\nAsk Question (type exit to quit): ")

    if question.lower() == "exit":
        break

    answer = ask_video(question)

    print("\nAnswer:")
    print(answer)


Ask Question (type exit to quit): what is RAg

Answer:
संदर्भ में RAG को एक "रैग पाइपलाइन" के रूप में संदर्भित किया गया है जिसमें निम्नलिखित चरण शामिल हैं:

*   इंडेक्सिंग
*   रिट्रीवल
*   आर्गुमेंटेशन (या ऑग्मेंटेशन)
*   जनरेशन

इस पाइपलाइन का लक्ष्य इन सभी चरणों को एक साथ जोड़ना है ताकि एक कंपोनेंट का आउटपुट स्वचालित रूप से अगले कंपोनेंट का इनपुट बन जाए और एक सिंगल इनवोक फंक्शन कॉल से पूरी पाइपलाइन स्वचालित रूप से निष्पादित हो जाए।

Ask Question (type exit to quit): quit

Answer:
The provided context does not contain information about "quit".
